# Using the CallableMultiplexer Class in baseobjects

## Introduction

The `CallableMultiplexer` class provides a flexible mechanism for selecting between different functions or methods to be used as a callable. It extends the `BaseMethod` class to create a callable object that can dynamically switch between different functions or methods at runtime.

This tutorial will guide you through:
- Understanding the purpose and functionality of the `CallableMultiplexer` class
- Creating and using a callable multiplexer
- Adding functions and methods to the multiplexer
- Selecting which function or method to use at runtime
- Understanding the differences between `CallableMultiplexer`, `FunctionMultiplexer`, and `MethodMultiplexer`
- Practical use cases for callable multiplexers

**Prerequisites:**
- Basic understanding of Python callables (functions and methods)
- Familiarity with Python's method binding mechanism
- Understanding of function objects in Python

### Table of Contents

- [Importing the Module](#Importing-the-Module)
- [Core Functionality](#Core-Functionality)
- [Module Interaction](#Module-Interaction)
- [Advanced Features](#Advanced-Features)
- [Examples](#Examples)
- [API Highlights](#API-Highlights)
- [Troubleshooting / FAQs](#Troubleshooting-/-FAQs)
- [Conclusion and Next Steps](#Conclusion-and-Next-Steps)

## Importing the Module

In [1]:
from baseobjects.functions import CallableMultiplexer, FunctionRegistry

## Core Functionality

The `CallableMultiplexer` class is designed to select between different functions or methods to be used as a callable. It provides a way to dynamically switch between different implementations at runtime. Let's explore the core functionality and understand how it works.

### Basic Concept

At its core, `CallableMultiplexer` is a callable object that:

1. Maintains a registry of functions/methods
2. Can be bound to an object instance
3. Selects a specific function/method to use when called
4. Handles the binding of methods to instances when necessary

Let's start with a simple example of creating and using a `CallableMultiplexer`:

In [2]:
# Create a simple function registry
registry = FunctionRegistry()


# Define some functions to add to the registry
def add(a, b):
    return a + b


def subtract(a, b):
    return a - b


def multiply(a, b):
    return a * b


# Add functions to the registry
registry["add"] = add
registry["subtract"] = subtract
registry["multiply"] = multiply

# Create a CallableMultiplexer with the registry
multiplexer = CallableMultiplexer(registry=registry)

# Select a function to use
multiplexer.select("add")

# Use the multiplexer as a callable
result = multiplexer(5, 3)
print(f"5 + 3 = {result}")

# Change the selected function
multiplexer.select("multiply")
result = multiplexer(5, 3)
print(f"5 * 3 = {result}")

5 + 3 = 8
5 * 3 = 15


In the example above, we created a simple function registry and added three functions to it. We then created a `CallableMultiplexer` with this registry and selected which function to use. The multiplexer can be called directly, and it will delegate the call to the selected function.

### Working with Methods

One of the powerful features of `CallableMultiplexer` is its ability to work with methods. When working with methods, the multiplexer needs to handle the binding of the method to an instance. This is controlled by the `is_binding_wrapper` flag.

Let's see how we can use `CallableMultiplexer` with methods:

In [3]:
# Create a class with methods
class MathOperations:
    def add(self, a, b):
        return a + b

    def subtract(self, a, b):
        return a - b

    def multiply(self, a, b):
        return a * b


# Create an instance of the class
math_ops = MathOperations()

# Create a registry and add methods from the instance
registry = FunctionRegistry()
registry["add"] = math_ops.add
registry["subtract"] = math_ops.subtract
registry["multiply"] = math_ops.multiply

# Create a CallableMultiplexer with the registry
multiplexer = CallableMultiplexer(registry=registry)

# Select a method to use
multiplexer.select("add")

# Try to use the multiplexer (this will fail because methods need binding)
try:
    result = multiplexer(5, 3)
    print(f"5 + 3 = {result}")
except TypeError as e:
    print(f"Error: {e}")

# Set is_binding_wrapper to True to enable method binding
multiplexer.is_binding_wrapper = True
multiplexer.bind_self(math_ops)

# Now try again
result = multiplexer(5, 3)
print(f"With binding enabled: 5 + 3 = {result}")

5 + 3 = 8
With binding enabled: 5 + 3 = 8


In this example, we see that when working with methods, we need to set `is_binding_wrapper` to `True` to enable method binding. This is because methods in Python are bound to an instance, and the multiplexer needs to handle this binding.

### Direct Instance Method Selection

Another powerful feature of `CallableMultiplexer` is its ability to select methods directly from a wrapped instance, without needing to add them to a registry first:

In [4]:
# Create a class with methods
class DataProcessor:
    def __init__(self) -> None:
        self.data = [1, 2, 3, 4, 5]

        # Create a CallableMultiplexer that wraps this instance
        self.multiplexer = CallableMultiplexer(instance=self)

        # Select a method to use
        self.multiplexer.select("sum_data")

    def sum_data(self):
        return sum(self.data)

    def average_data(self):
        return sum(self.data) / len(self.data)

    def process(self):
        # Use the multiplexer to call the selected method
        return self.multiplexer()

    def set_processor(self, method_name) -> None:
        # Change the selected method
        self.multiplexer.select(method_name)


# Create an instance of the class
processor = DataProcessor()

# Use the default processor (sum_data)
result = processor.process()
print(f"Sum of data: {result}")

# Change the processor to average_data
processor.set_processor("average_data")
result = processor.process()
print(f"Average of data: {result}")

Sum of data: 15
Average of data: 3.0


In this example, we created a `CallableMultiplexer` that wraps an instance of `DataProcessor`. We can then select methods directly from the instance without needing to add them to a registry first. This is a powerful feature that allows for dynamic method selection at runtime.

## Module Interaction

The `CallableMultiplexer` class is part of the `baseobjects.functions` module and interacts with other components of the baseobjects package. It extends the `BaseMethod` class from the `baseobjects.bases` module and uses the `FunctionRegistry` class for storing functions and methods.

Let's see how `CallableMultiplexer` interacts with other components:

In [5]:
# Create a custom multiplexer that extends CallableMultiplexer
class TypedCallableMultiplexer(CallableMultiplexer):
    """A callable multiplexer that only accepts functions with specific parameter types."""

    def __init__(self, param_type=None, *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        self.param_type = param_type

    def select(self, name) -> None:
        """Override select to check parameter types."""
        if name is None:
            super().select(name)
            return

        # Get the function from the registry or instance
        if (func := self.registry.get(name, None)) is not None:
            pass
        elif self._self_ is not None:
            func = getattr(self._self_(), name)
        else:
            msg = f"Function '{name}' not found"
            raise ValueError(msg)

        # Check parameter types if specified
        if self.param_type is not None and hasattr(func, "__annotations__"):
            for param_name, param_type in func.__annotations__.items():
                if param_name != "return" and param_type != self.param_type:
                    msg = f"Function '{name}' parameter '{param_name}' is not of type {self.param_type.__name__}"
                    raise TypeError(msg)

        # Call the parent select method
        super().select(name)


# Create functions with type annotations
def int_add(a: int, b: int) -> int:
    return a + b


def float_add(a: float, b: float) -> float:
    return a + b


def str_concat(a: str, b: str) -> str:
    return a + b


# Create a registry and add the functions
registry = FunctionRegistry()
registry["int_add"] = int_add
registry["float_add"] = float_add
registry["str_concat"] = str_concat

# Create a TypedCallableMultiplexer that only accepts functions with int parameters
int_multiplexer = TypedCallableMultiplexer(param_type=int, registry=registry)

# Select a function with int parameters
int_multiplexer.select("int_add")
result = int_multiplexer(5, 3)
print(f"int_add(5, 3) = {result}")

# Try to select a function with float parameters
try:
    int_multiplexer.select("float_add")
except TypeError as e:
    print(f"Error: {e}")

# Try to select a function with str parameters
try:
    int_multiplexer.select("str_concat")
except TypeError as e:
    print(f"Error: {e}")

int_add(5, 3) = 8
Error: Function 'float_add' parameter 'a' is not of type int
Error: Function 'str_concat' parameter 'a' is not of type int


In this example, we created a custom multiplexer that extends `CallableMultiplexer` to only accept functions with specific parameter types. This demonstrates how `CallableMultiplexer` can be extended and integrated with other components of the baseobjects package.

## Advanced Features

### Understanding the Relationship Between CallableMultiplexer, FunctionMultiplexer, and MethodMultiplexer

The baseobjects package provides three types of multiplexers:

1. `CallableMultiplexer`: The base class that can work with both functions and methods
2. `FunctionMultiplexer`: A subclass specialized for working with standalone functions
3. `MethodMultiplexer`: A subclass specialized for working with methods

Let's explore the differences between these three multiplexers:

In [6]:
from baseobjects.functions import FunctionMultiplexer, MethodMultiplexer


# Create a class with methods
class MathOperations:
    def add(self, a, b):
        return a + b

    def subtract(self, a, b):
        return a - b


# Create an instance of the class
math_ops = MathOperations()

# Create a registry and add methods from the instance
registry = FunctionRegistry()
registry["add"] = math_ops.add
registry["subtract"] = math_ops.subtract


# Create standalone functions
def standalone_add(a, b):
    return a + b


def standalone_subtract(a, b):
    return a - b


# Add standalone functions to the registry
registry["standalone_add"] = standalone_add
registry["standalone_subtract"] = standalone_subtract

# Create the three types of multiplexers
callable_multiplexer = CallableMultiplexer(registry=registry, instance=math_ops)
function_multiplexer = FunctionMultiplexer(registry=registry)
method_multiplexer = MethodMultiplexer(registry=registry, instance=math_ops)

# Test with methods
print("Testing with methods:")

# CallableMultiplexer with is_binding_wrapper=False (default after select)
callable_multiplexer.select("add")
try:
    result = callable_multiplexer(5, 3)
    print(f"CallableMultiplexer: 5 + 3 = {result}")
except TypeError as e:
    print(f"CallableMultiplexer error: {e}")

# CallableMultiplexer with is_binding_wrapper=True
callable_multiplexer.is_binding_wrapper = True
result = callable_multiplexer(5, 3)
print(f"CallableMultiplexer (with binding): 5 + 3 = {result}")

# FunctionMultiplexer
function_multiplexer.select("add")
try:
    result = function_multiplexer(5, 3)
    print(f"FunctionMultiplexer: 5 + 3 = {result}")
except TypeError as e:
    print(f"FunctionMultiplexer error: {e}")

# MethodMultiplexer
method_multiplexer.select("add")
result = method_multiplexer(5, 3)
print(f"MethodMultiplexer: 5 + 3 = {result}")

# Test with standalone functions
print("\nTesting with standalone functions:")

# CallableMultiplexer with is_binding_wrapper=False
callable_multiplexer.select("standalone_add")
callable_multiplexer.is_binding_wrapper = False
result = callable_multiplexer(5, 3)
print(f"CallableMultiplexer: 5 + 3 = {result}")

# FunctionMultiplexer
function_multiplexer.select("standalone_add")
result = function_multiplexer(5, 3)
print(f"FunctionMultiplexer: 5 + 3 = {result}")

# MethodMultiplexer
method_multiplexer.select("standalone_add")
try:
    result = method_multiplexer(5, 3)
    print(f"MethodMultiplexer: 5 + 3 = {result}")
except TypeError as e:
    print(f"MethodMultiplexer error: {e}")

Testing with methods:
CallableMultiplexer: 5 + 3 = 8
CallableMultiplexer (with binding): 5 + 3 = 8
FunctionMultiplexer: 5 + 3 = 8
MethodMultiplexer: 5 + 3 = 8

Testing with standalone functions:
CallableMultiplexer: 5 + 3 = 8
FunctionMultiplexer: 5 + 3 = 8
MethodMultiplexer error: standalone_add() takes 2 positional arguments but 3 were given


This example demonstrates the key differences between the three multiplexer types:

1. `CallableMultiplexer`:
   - Can work with both methods and standalone functions
   - Needs `is_binding_wrapper=True` to work with methods
   - Needs `is_binding_wrapper=False` to work with standalone functions

2. `FunctionMultiplexer`:
   - Specialized for working with standalone functions
   - Does not bind methods to instances
   - Will fail if used with methods that expect a `self` parameter

3. `MethodMultiplexer`:
   - Specialized for working with methods
   - Automatically binds methods to the provided instance
   - May have issues with standalone functions that don't expect a `self` parameter

### Dynamic Method Addition and Selection

One advanced use case for `CallableMultiplexer` is dynamic method addition and selection at runtime:

In [7]:
import types


# Create a class with dynamic method addition
class DynamicProcessor:
    def __init__(self) -> None:
        self.data = [1, 2, 3, 4, 5]

        # Create a CallableMultiplexer that wraps this instance
        self.multiplexer = CallableMultiplexer(instance=self)
        self.multiplexer.is_binding_wrapper = True

        # Select a method to use
        self.multiplexer.select("sum_data")

    def sum_data(self):
        return sum(self.data)

    def process(self):
        # Use the multiplexer to call the selected method
        return self.multiplexer()

    def add_processor(self, name, func) -> None:
        # Add a new method to the instance
        setattr(self, name, types.MethodType(func, self))

    def set_processor(self, method_name) -> None:
        # Change the selected method
        self.multiplexer.select(method_name)


# Create an instance of the class
processor = DynamicProcessor()

# Use the default processor (sum_data)
result = processor.process()
print(f"Sum of data: {result}")


# Add a new processor dynamically
def average_data(self):
    return sum(self.data) / len(self.data)


processor.add_processor("average_data", average_data)

# Select and use the new processor
processor.set_processor("average_data")
result = processor.process()
print(f"Average of data: {result}")


# Add another processor dynamically
def product_data(self):
    result = 1
    for item in self.data:
        result *= item
    return result


processor.add_processor("product_data", product_data)

# Select and use the new processor
processor.set_processor("product_data")
result = processor.process()
print(f"Product of data: {result}")

Sum of data: 15
Average of data: 3.0
Product of data: 120


This example demonstrates how `CallableMultiplexer` can be used to dynamically add and select methods at runtime. This is a powerful feature that allows for flexible and extensible code.

## Examples

### Example 1: Strategy Pattern Implementation

The Strategy pattern is a behavioral design pattern that lets you define a family of algorithms, put each of them into a separate class, and make their objects interchangeable. Let's implement a simple strategy pattern using `CallableMultiplexer`:

In [8]:
# Create a class that uses CallableMultiplexer to implement the Strategy pattern
class TextProcessor:
    def __init__(self) -> None:
        # Create a registry for our text processing strategies
        self.registry = FunctionRegistry()

        # Add some text processing strategies
        self.registry["uppercase"] = lambda text: text.upper()
        self.registry["lowercase"] = lambda text: text.lower()
        self.registry["capitalize"] = lambda text: text.title()
        self.registry["reverse"] = lambda text: text[::-1]

        # Create a CallableMultiplexer with our registry
        self.strategy = CallableMultiplexer(registry=self.registry)

        # Set a default strategy
        self.strategy.select("uppercase")

    def process(self, text):
        """Process the text using the current strategy."""
        return self.strategy(text)

    def set_strategy(self, strategy_name) -> None:
        """Set the text processing strategy."""
        if strategy_name not in self.registry:
            msg = f"Unknown strategy: {strategy_name}"
            raise ValueError(msg)

        self.strategy.select(strategy_name)


# Create a text processor
processor = TextProcessor()

# Process some text with the default strategy (uppercase)
text = "Hello, world!"
result = processor.process(text)
print(f"Default strategy (uppercase): '{result}'")

# Change the strategy and process again
processor.set_strategy("lowercase")
result = processor.process(text)
print(f"Changed strategy to lowercase: '{result}'")

# Try other strategies
processor.set_strategy("capitalize")
result = processor.process(text)
print(f"Changed strategy to capitalize: '{result}'")

processor.set_strategy("reverse")
result = processor.process(text)
print(f"Changed strategy to reverse: '{result}'")

# Add a new strategy dynamically
processor.registry["count_chars"] = lambda text: f"{text} ({len(text)} characters)"
processor.set_strategy("count_chars")
result = processor.process(text)
print(f"Added and selected new strategy (count_chars): '{result}'")

Default strategy (uppercase): 'HELLO, WORLD!'
Changed strategy to lowercase: 'hello, world!'
Changed strategy to capitalize: 'Hello, World!'
Changed strategy to reverse: '!dlrow ,olleH'
Added and selected new strategy (count_chars): 'Hello, world! (13 characters)'


### Example 2: Command Pattern Implementation

The Command pattern is a behavioral design pattern that turns a request into a stand-alone object containing all information about the request. Let's implement a simple command pattern using `CallableMultiplexer`:

In [9]:
# Create a class that represents a document
class Document:
    def __init__(self) -> None:
        self.content = ""
        self.filename = "untitled.txt"

    def __str__(self) -> str:
        return f"Document: {self.filename}\nContent: {self.content}"


# Create a class that uses CallableMultiplexer to implement the Command pattern
class DocumentEditor:
    def __init__(self) -> None:
        self.document = Document()

        # Create a registry for our commands
        self.registry = FunctionRegistry()

        # Add some commands
        self.registry["new"] = self.new_document
        self.registry["open"] = self.open_document
        self.registry["save"] = self.save_document
        self.registry["append"] = self.append_text

        # Create a CallableMultiplexer with our registry
        self.command = CallableMultiplexer(registry=self.registry, instance=self)
        self.command.is_binding_wrapper = True

        # Command history
        self.history = []

    def execute(self, command_name, *args, **kwargs):
        """Execute a command by name."""
        if command_name not in self.registry:
            msg = f"Unknown command: {command_name}"
            raise ValueError(msg)

        # Select the command
        self.command.select(command_name)

        # Execute the command
        result = self.command(*args, **kwargs)

        # Record the command in history
        self.history.append((command_name, args, kwargs))

        return result

    def new_document(self) -> str:
        """Create a new document."""
        self.document = Document()
        return "Created new document"

    def open_document(self, filename) -> str:
        """Open a document from a file."""
        self.document.filename = filename
        self.document.content = f"Content of {filename}"  # Simulated file loading
        return f"Opened document: {filename}"

    def save_document(self) -> str:
        """Save the document to a file."""
        return f"Saved document to {self.document.filename}"

    def append_text(self, text) -> str:
        """Append text to the document."""
        self.document.content += text
        return f"Appended text: '{text}'"

    def show_history(self) -> None:
        """Show the command execution history."""
        print("Command History:")
        for i, (cmd, args, kwargs) in enumerate(self.history, 1):
            args_str = ", ".join(repr(arg) for arg in args)
            kwargs_str = ", ".join(f"{k}={v!r}" for k, v in kwargs.items())
            all_args = ", ".join(filter(None, [args_str, kwargs_str]))
            print(f"{i}. {cmd}({all_args})")


# Create a document editor
editor = DocumentEditor()

# Execute some commands
print(editor.execute("new"))
print(editor.execute("append", "Hello, world!"))
print(editor.execute("save"))
print(editor.execute("open", "example.txt"))
print(editor.execute("append", " Additional text."))

# Show the document
print("\nCurrent document:")
print(editor.document)

# Show command history
print("\nCommand history:")
editor.show_history()

Created new document
Appended text: 'Hello, world!'
Saved document to untitled.txt
Opened document: example.txt
Appended text: ' Additional text.'

Current document:
Document: example.txt
Content: Content of example.txt Additional text.

Command history:
Command History:
1. new()
2. append('Hello, world!')
3. save()
4. open('example.txt')
5. append(' Additional text.')


## API Highlights

The `CallableMultiplexer` class provides the following key features:

- **Function/Method Selection**: Select between different functions or methods at runtime
- **Registry Integration**: Use a `FunctionRegistry` to store and manage functions/methods
- **Instance Binding**: Bind methods to instances when necessary
- **Dynamic Selection**: Select methods directly from a wrapped instance

Key methods:
- `__init__(registry=None, instance=None, owner=None, select=None, binding=False, ...)`: Constructor with options for initial setup
- `select(name)`: Select a function/method to use
- `add_function(name, func)`: Add a function to the registry
- `add_method(name, method)`: Add a method to the registry
- `add_select_function(name, func)`: Add a function to the registry and select it
- `add_select_method(name, method)`: Add a method to the registry and select it
- `bind_selected(instance=None, owner=None)`: Bind the selected function to an instance

For the full API documentation, refer to the baseobjects documentation.

## Troubleshooting / FAQs

### Q: Why am I getting a TypeError when using CallableMultiplexer with methods?

A: When using `CallableMultiplexer` with methods, you need to set `is_binding_wrapper` to `True` to enable method binding. This is because methods in Python are bound to an instance, and the multiplexer needs to handle this binding.

```python
# Create a CallableMultiplexer with a registry containing methods
multiplexer = CallableMultiplexer(registry=registry)
multiplexer.select('some_method')

# Set is_binding_wrapper to True to enable method binding
multiplexer.is_binding_wrapper = True

# Now you can call the multiplexer with methods
result = multiplexer(arg1, arg2)
```

### Q: How do I use CallableMultiplexer with standalone functions?

A: When using `CallableMultiplexer` with standalone functions, you should set `is_binding_wrapper` to `False` (which is the default after calling `select()`).

```python
# Create a CallableMultiplexer with a registry containing standalone functions
multiplexer = CallableMultiplexer(registry=registry)
multiplexer.select('some_function')

# Ensure is_binding_wrapper is False (default after select)
multiplexer.is_binding_wrapper = False

# Now you can call the multiplexer with standalone functions
result = multiplexer(arg1, arg2)
```

### Q: When should I use CallableMultiplexer vs. FunctionMultiplexer vs. MethodMultiplexer?

A: Choose the multiplexer based on your specific needs:

- Use `CallableMultiplexer` when you need flexibility to work with both functions and methods, or when you need to dynamically switch between binding and non-binding behavior.
- Use `FunctionMultiplexer` when you're only working with standalone functions and don't need method binding.
- Use `MethodMultiplexer` when you're only working with methods and need automatic binding to an instance.

### Q: How do I select methods directly from a wrapped instance?

A: You can select methods directly from a wrapped instance by providing the instance to the constructor and using the `select` method with the name of the method:

```python
# Create a CallableMultiplexer that wraps an instance
multiplexer = CallableMultiplexer(instance=my_instance)

# Select a method from the instance
multiplexer.select('some_method')

# Set is_binding_wrapper to True for method binding
multiplexer.is_binding_wrapper = True

# Now you can call the multiplexer
result = multiplexer(arg1, arg2)
```

## Conclusion and Next Steps

In this tutorial, we've explored the `CallableMultiplexer` class from the baseobjects package. We've learned how to use it to create a callable that can dynamically select between different functions or methods at runtime.

Key takeaways:
- `CallableMultiplexer` provides a flexible mechanism for selecting between different functions or methods
- It can work with both standalone functions and methods, with appropriate binding configuration
- It integrates with `FunctionRegistry` for storing and managing functions/methods
- It can be used to implement design patterns like Strategy and Command
- It has specialized subclasses (`FunctionMultiplexer` and `MethodMultiplexer`) for specific use cases

Next steps:
- Explore the specialized multiplexer classes (`FunctionMultiplexer` and `MethodMultiplexer`)
- Combine multiplexers with other components of the baseobjects package
- Create your own custom multiplexers by subclassing `CallableMultiplexer`
- Check out the examples directory for more examples of using multiplexers

For more information, refer to the baseobjects documentation and examples.